[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/bayesian_networks_lucada.ipynb)

# Bayesian Networks for Clinical Decision Support

This notebook accompanies the blog post [Bayesian Networks for Clinical Decision Support](https://sesen.ai/blog/bayesian-networks-clinical-decision-support) (Bishop PRML chapter 8, post BP7).

We build a small Bayesian network inspired by the LUCADA-trained network in [Sesen et al. (2014)](https://royalsocietypublishing.org/doi/10.1098/rsif.2014.0534) (Royal Society Interface). The toy network has seven discrete nodes (Smoking, Age, Histology, PerformanceStatus, TNM, Treatment, Survival) and nine directed edges. We use [pgmpy](https://pgmpy.org/) for the model, sampling, parameter learning, exact inference (variable elimination), and structure learning (hill climbing with BIC).

All synthetic data is generated by forward sampling from a hand-specified set of CPTs; nothing here is trained on real LUCADA data.

## 1. Setup

In [ ]:
# !pip install pgmpy --quiet   # uncomment in Colab
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.parameter_estimator import DiscreteMLE, DiscreteBayesianEstimator
from pgmpy.estimators import HillClimbSearch, BIC
from pgmpy.sampling import BayesianModelSampling

np.random.seed(0)

## 2. Define the toy lung-cancer DAG

Variable encoding:

| Variable | 0 | 1 | 2 |
|---|---|---|---|
| Smoking | no | yes |  |
| Age | ≤60 | >60 |  |
| Histology | NSCLC | SCLC |  |
| PerfStatus | good (0–1) | poor (2–4) |  |
| TNM | early (I–II) | late (III–IV) |  |
| Treatment | surgery | chemo-radio | palliative |
| Survival | alive at 1 year | dead at 1 year |  |


In [ ]:
EDGES = [
    ('Smoking', 'Histology'),
    ('Smoking', 'PerfStatus'),
    ('Age', 'PerfStatus'),
    ('PerfStatus', 'Treatment'),
    ('TNM', 'Treatment'),
    ('Histology', 'Treatment'),
    ('TNM', 'Survival'),
    ('Treatment', 'Survival'),
    ('PerfStatus', 'Survival'),
]
model = DiscreteBayesianNetwork(EDGES)

### 2.1 Conditional probability tables

Seven CPTs, one per node, totalling 35 free parameters.
(A fully connected discrete joint over the same variables would need `2*2*2*2*2*3*2 - 1 = 191` parameters.)

In [ ]:
cpd_smoking = TabularCPD('Smoking', 2, [[0.40], [0.60]])
cpd_age     = TabularCPD('Age',     2, [[0.45], [0.55]])
cpd_tnm     = TabularCPD('TNM',     2, [[0.45], [0.55]])

cpd_hist = TabularCPD(
    'Histology', 2,
    [[0.85, 0.78],
     [0.15, 0.22]],
    evidence=['Smoking'], evidence_card=[2],
)

cpd_ps = TabularCPD(
    'PerfStatus', 2,
    [[0.85, 0.65, 0.70, 0.40],
     [0.15, 0.35, 0.30, 0.60]],
    evidence=['Smoking', 'Age'], evidence_card=[2, 2],
)

cpd_tx = TabularCPD(
    'Treatment', 3,
    [[0.70, 0.10, 0.10, 0.05, 0.20, 0.05, 0.05, 0.02],
     [0.25, 0.80, 0.75, 0.80, 0.30, 0.20, 0.25, 0.18],
     [0.05, 0.10, 0.15, 0.15, 0.50, 0.75, 0.70, 0.80]],
    evidence=['PerfStatus', 'TNM', 'Histology'], evidence_card=[2, 2, 2],
)

alive, dead = [], []
for tnm in range(2):
    for tx in range(3):
        for ps in range(2):
            base = 0.85 if tnm == 0 else 0.45
            tx_adj = {0: +0.05, 1: 0.00, 2: -0.20}[tx]
            ps_adj = -0.15 if ps == 1 else 0.00
            p_alive = max(0.05, min(0.95, base + tx_adj + ps_adj))
            alive.append(p_alive)
            dead.append(1 - p_alive)
cpd_surv = TabularCPD(
    'Survival', 2, [alive, dead],
    evidence=['TNM', 'Treatment', 'PerfStatus'], evidence_card=[2, 3, 2],
)

model.add_cpds(cpd_smoking, cpd_age, cpd_tnm, cpd_hist, cpd_ps, cpd_tx, cpd_surv)
assert model.check_model()
print('Total free parameters:', sum(cpd.values.size - cpd.values.shape[0]
                                  for cpd in model.get_cpds()))

### 2.2 Visualise the DAG
The graph encodes the factorisation `p(x) = product over k of p(x_k | parents(x_k))`.

In [ ]:
G = nx.DiGraph()
G.add_edges_from(EDGES)
pos = {
    'Smoking':    (-2.6,  2.0),
    'Age':        (-2.6,  0.0),
    'Histology':  (-0.6,  2.0),
    'PerfStatus': (-0.6,  0.0),
    'TNM':        (-0.6, -2.0),
    'Treatment':  ( 1.6,  0.6),
    'Survival':   ( 4.0,  0.0),
}
fig, ax = plt.subplots(figsize=(10, 5.5))
nx.draw_networkx_edges(G, pos, ax=ax, arrowsize=22, width=1.6, node_size=4800,
                       min_source_margin=14, min_target_margin=18)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color='white',
                       edgecolors='#1f4e79', linewidths=2.2, node_size=4800)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=11, font_weight='bold',
                        font_color='#1f4e79')
ax.axis('off')
plt.title('Toy lung-cancer Bayesian network (Bishop §8.1)')
plt.show()

## 3. Forward-sample synthetic patient data
Ancestral sampling (Bishop §8.1.2): walk the topological order, sample each node given its already-sampled parents.

In [ ]:
sampler = BayesianModelSampling(model)
data = sampler.forward_sample(size=4000, show_progress=False)
print('shape:', data.shape)
print('survival rate (alive):', (data.Survival == 0).mean().round(3))
print('treatment distribution:')
data.Treatment.value_counts(normalize=True).round(3)

## 4. Parameter learning from the synthetic data

### 4.1 Maximum likelihood

For discrete CPTs with all parents observed, MLE is just empirical conditional frequency.

In [ ]:
empty = DiscreteBayesianNetwork(EDGES)
empty.fit(data, estimator=DiscreteMLE())
print('Recovered P(good PS | Smoking, Age):')
print(empty.get_cpds('PerfStatus').values[0].round(3))
print('Ground truth:')
print(cpd_ps.values[0].round(3))

### 4.2 Bayesian estimation with a BDeu prior

The Dirichlet prior turns the closed-form update into `p_hat = (count + alpha) / (N_pa + alpha * K)`. With `equivalent_sample_size = 10`, the prior contributes the equivalent of 10 uniformly-distributed pseudo-patients.

In [ ]:
bayes = DiscreteBayesianNetwork(EDGES)
bayes.fit(
    data,
    estimator=DiscreteBayesianEstimator(prior_type='BDeu',
                                        equivalent_sample_size=10),
)
print(bayes.get_cpds('PerfStatus').values[0].round(3))

## 5. Inference

### 5.1 Marginals and conditionals via variable elimination


In [ ]:
infer = VariableElimination(model)

print('P(Survival):', infer.query(['Survival'], show_progress=False).values.round(3))

profile = {'TNM': 1, 'Histology': 0, 'PerfStatus': 0}
print('Late-stage NSCLC, good PS:')
for tx, name in [(0, 'surgery'), (1, 'chemo-radio'), (2, 'palliative')]:
    p = infer.query(['Survival'],
                    evidence={**profile, 'Treatment': tx},
                    show_progress=False).values
    print(f'  do(Treatment={name:11s}): P(alive at 1y) = {p[0]:.3f}')

### 5.2 Explaining away (collider conditioning)

Survival is a head-to-head node with parents PerfStatus, TNM, Treatment. Conditioning on Survival creates dependence between its parents.

In [ ]:
p1 = infer.query(['PerfStatus'], evidence={'Survival': 1},
                 show_progress=False).values
p2 = infer.query(['PerfStatus'], evidence={'Survival': 1, 'Treatment': 2},
                 show_progress=False).values
print(f'P(poor PS | dead)                       = {p1[1]:.3f}')
print(f'P(poor PS | dead, Treatment=palliative) = {p2[1]:.3f}')

### 5.3 Counterfactual treatment plot

In [ ]:
vals = []
for tx in [None, 0, 1, 2]:
    ev = dict(profile)
    if tx is not None:
        ev['Treatment'] = tx
    vals.append(infer.query(['Survival'], evidence=ev,
                            show_progress=False).values[0])
labels = ['BN pick', 'do(surgery)', 'do(chemo-radio)', 'do(palliative)']
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, vals, color=['#1f4e79', '#2e7d32', '#ed7d31', '#b71c1c'])
for i, v in enumerate(vals):
    ax.text(i, v + 0.015, f'{v:.2f}', ha='center', fontweight='bold')
ax.set_ylabel('P(alive at 1 year)')
ax.set_ylim(0, 1)
ax.spines[['top', 'right']].set_visible(False)
plt.title('Counterfactual treatment effect (late-stage NSCLC, good PS)')
plt.show()

## 6. Structure learning with hill climbing

Score-based search: start from the empty graph, evaluate every legal single-edge add/remove/reverse, accept the move with the largest BIC improvement, repeat. We use a 5-node subset for clarity.

In [ ]:
sub = ['Smoking', 'PerfStatus', 'TNM', 'Treatment', 'Survival']
sub_data = data[sub]
hc = HillClimbSearch(sub_data)
learned = hc.estimate(scoring_method=BIC(sub_data), max_indegree=4,
                      show_progress=False)
print('Learned edges:')
for e in learned.edges():
    print(f'  {e[0]} -> {e[1]}')

Compare to the true edges in the subset: PerfStatus → Treatment, TNM → Treatment, PerfStatus → Survival, TNM → Survival, Treatment → Survival, Smoking → PerfStatus. The hill-climber recovers the skeleton but, as discussed in the post, observational data alone cannot orient every edge: some recovered directions belong to the same Markov-equivalence class as the truth.

## 7. Markov equivalence

`A → B → C`, `A ← B → C`, and `A ← B ← C` all imply `A ⫫ C | B` and no other independence. They are observationally indistinguishable. To break the tie you need expert priors (temporal ordering, biological constraints) or experimental interventions.

## 8. Exercises

1. **Continuous variables.** Replace `Age` with a continuous variable sampled    from a Gaussian. Discretise it into three bins and refit. How sensitive are    the recovered CPTs to the bin boundaries?
2. **Sample size sensitivity.** Run the structure-learning step with 100, 500,    1,000, and 5,000 samples. At what point does hill climbing recover the true    skeleton?
3. **Constraint-based search.** Replace `HillClimbSearch` with `PC`    (`from pgmpy.estimators import PC`) and compare the partially-directed graph    it returns to the score-based output.
4. **Causal direction.** Add a Smoking → Age edge to the model (biologically    nonsensical, but the data will fit it). Refit and query; which inferences    change? Which stay the same?
5. **Real data.** Download the [ASIA network](https://www.bnlearn.com/bnrepository/discrete-small.html#asia)    from bnlearn, load it into pgmpy, and run all of the above on a published    benchmark.
